# REGEN-R0 — regenerate all five legacy candidate families
This notebook is an execution orchestrator. Retrieval logic stays in the repository scripts/modules. It stops after validated legacy artifacts; it does **not** run the A2 canonical union.

In [ ]:
from pathlib import Path

REPO_ROOT = Path('/kaggle/working/Amazon_ML_Challenge_2026')  # edit to the cloned repository
DATA_DIR = Path('/kaggle/input/amazon-ml-processed/data/processed')  # contains train/ and test/
OUTPUT_ROOT = Path('/kaggle/working/legacy_candidates_regenerated')
SPLIT = 'train'  # run train successfully before changing this to test

RUN_EXACT = True
RUN_CHAR = True
RUN_WORD = True
RUN_STRUCTURED = True
RUN_DENSE = True
RUN_SMOKE = True
RUN_FULL = True
FORCE_REBUILD = False
SMOKE_QUERY_COUNT = 256
SMOKE_SEED = 2026
SPARSE_CHUNK_SIZE = 10_000
MAX_MEMORY_FRACTION = 0.75
MAX_STRUCTURED_BLOCK_PAIRS = 50_000_000

assert SPLIT in {'train', 'test'}
assert REPO_ROOT.is_dir(), REPO_ROOT
import sys
sys.path.insert(0, str(REPO_ROOT / 'code/business_entity_resolution/src'))

## 1. Environment and deterministic FAISS resolution

In [ ]:
# Do not install faiss-cpu and faiss-gpu together. This notebook uses the existing Kaggle
# FAISS build and records whether it exposes GPU APIs. If imports below fail, restart with
# exactly one FAISS distribution installed (prefer the Kaggle-compatible GPU build).
import importlib.metadata as md, json, os, platform, shutil, subprocess
installed_faiss = []
for distribution in ('faiss-cpu', 'faiss-gpu', 'faiss-gpu-cu12'):
    try:
        installed_faiss.append((distribution, md.version(distribution)))
    except md.PackageNotFoundError:
        pass
if len(installed_faiss) > 1:
    raise RuntimeError(f'Conflicting FAISS distributions installed: {installed_faiss}')

from business_entity_resolution.regeneration import environment_report
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
env = environment_report(REPO_ROOT, OUTPUT_ROOT)
env['faiss_distributions'] = installed_faiss
for package, module_name in [('Polars','polars'), ('PyArrow','pyarrow'), ('NumPy','numpy'),
                             ('SciPy','scipy'), ('scikit-learn','sklearn'),
                             ('sentence-transformers','sentence_transformers'), ('FAISS','faiss'),
                             ('psutil','psutil')]:
    module = __import__(module_name)
    env[package] = getattr(module, '__version__', 'unknown')
print(json.dumps(env, indent=2))
subprocess.run(['nvidia-smi'], check=False)
if not env['gpu']['available']:
    raise RuntimeError('Select a Kaggle GPU accelerator before dense regeneration')
(OUTPUT_ROOT / 'logs').mkdir(exist_ok=True)
(OUTPUT_ROOT / 'logs' / 'environment.json').write_text(json.dumps(env, indent=2), encoding='utf-8')

## 2. Verify processed inputs — stop instead of silently preparing data

In [ ]:
from business_entity_resolution.regeneration import verify_processed_inputs
processed_report = verify_processed_inputs(DATA_DIR, SPLIT)
print(json.dumps(processed_report, indent=2))

## 3. Versioned output root, runner, validation, manifests, and resume checks

In [ ]:
import psutil, shlex, threading, time
from business_entity_resolution.regeneration import (atomic_json, build_manifest, manifest_matches, validate_candidate)

for directory in ('train', 'test', 'dense_artifacts', 'manifests', 'logs', 'smoke'):
    (OUTPUT_ROOT / directory).mkdir(parents=True, exist_ok=True)

def run_logged(command, log_name):
    print('$', shlex.join(map(str, command)))
    log_path = OUTPUT_ROOT / 'logs' / log_name
    started = time.time(); peak = {'rss': 0}; stop_sampling = threading.Event()
    with log_path.open('w', encoding='utf-8') as log:
        process = subprocess.Popen(list(map(str, command)), cwd=REPO_ROOT, text=True,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
        root = psutil.Process(process.pid)
        def sample_memory():
            while not stop_sampling.wait(0.5):
                try:
                    rss = root.memory_info().rss + sum(child.memory_info().rss for child in root.children(recursive=True))
                    peak['rss'] = max(peak['rss'], rss)
                except psutil.Error:
                    pass
        sampler = threading.Thread(target=sample_memory, daemon=True); sampler.start()
        for line in process.stdout:
            print(line, end=''); log.write(line); log.flush()
        code = process.wait()
        stop_sampling.set(); sampler.join()
    if code:
        raise subprocess.CalledProcessError(code, command)
    return {'runtime_seconds': time.time() - started, 'peak_rss_bytes': peak['rss'], 'log': str(log_path)}

def execute_candidate_stage(*, label, retriever, script, command, output, config, input_report, dense=None):
    manifest_path = OUTPUT_ROOT / 'manifests' / f'{label}.json'
    input_identity = {key:value['identity'] for key,value in input_report['files'].items()}
    if manifest_matches(manifest_path, output, config, retriever, input_identity) and not FORCE_REBUILD:
        print(f'REUSE validated stage: {label}')
        return validate_candidate(output, retriever)
    if Path(output).exists() and not FORCE_REBUILD:
        raise RuntimeError(f'{output} exists but its manifest/config is invalid; inspect it or set FORCE_REBUILD=True')
    metrics = run_logged(command, f'{label}.log')
    validation = validate_candidate(output, retriever)
    peak_gpu = None; dense_payload = dense
    dense_log_path = OUTPUT_ROOT/'dense_artifacts'/SPLIT/'dense_run_log.json'
    if dense and dense_log_path.is_file():
        dense_log = json.loads(dense_log_path.read_text(encoding='utf-8'))
        peak_gpu = dense_log.get('gpu_peak_memory_bytes')
        dense_payload = {**dense, 'runtime_log':dense_log}
    manifest = build_manifest(artifact_type='legacy_candidates', split=SPLIT, retriever=retriever,
        source_script=script, repo_root=REPO_ROOT, processed_report=input_report, config=config,
        validation=validation, runtime_seconds=metrics['runtime_seconds'],
        peak_rss_bytes=metrics['peak_rss_bytes'], peak_gpu_memory_bytes=peak_gpu, dense=dense_payload)
    atomic_json(manifest_path, manifest)
    print(json.dumps(validation, indent=2))
    return validation

## 4. Deterministic Source-1 query smoke inputs

In [ ]:
from business_entity_resolution.regeneration import deterministic_smoke_ids, prepare_smoke_inputs
smoke_data_dir = OUTPUT_ROOT / 'smoke' / 'data'
smoke_ids = deterministic_smoke_ids(DATA_DIR / SPLIT / f'{SPLIT}_source1.parquet', SMOKE_QUERY_COUNT, SMOKE_SEED)
smoke_paths = prepare_smoke_inputs(DATA_DIR, smoke_data_dir, SPLIT, smoke_ids)
smoke_report = verify_processed_inputs(smoke_data_dir, SPLIT)
print(f'Smoke uses {len(smoke_ids)} Source-1 query IDs and the complete S2/S3 corpus')
print(json.dumps(smoke_paths, indent=2))

## 5. Prove bounded sparse top-K equivalence on deterministic data

In [ ]:
# Per-query top-K is independent across query chunks. This executable check compares one-shot
# sparse_dot_topn output with the bounded iterator using identical matrices and thresholds.
import numpy as np, scipy.sparse as sp
from business_entity_resolution.regeneration import _topk_callable, iter_sparse_topk
rng = np.random.default_rng(2026)
q = sp.random(37, 113, density=0.12, random_state=rng, format='csr', dtype=np.float32)
c_t = sp.random(113, 71, density=0.15, random_state=rng, format='csr', dtype=np.float32)
legacy_fn, impl = _topk_callable(7)
legacy = legacy_fn(q, c_t).tocoo()
legacy_rows = sorted(zip(legacy.row.tolist(), legacy.col.tolist(), legacy.data.tolist()))
bounded_rows = []
for start, rows, cols, scores in iter_sparse_topk(q, c_t, 7, 9):
    bounded_rows.extend(zip((rows + start).tolist(), cols.tolist(), scores.tolist()))
bounded_rows.sort()
assert [(r,c) for r,c,_ in legacy_rows] == [(r,c) for r,c,_ in bounded_rows]
assert np.allclose([s for *_,s in legacy_rows], [s for *_,s in bounded_rows], rtol=1e-6, atol=1e-7)
print(f'PASS: bounded chunking is pair/score-equivalent to one-shot {impl}')

## 6. Smoke all five retrievers (full corpus, selected query IDs only)

In [ ]:
PYTHON = sys.executable
smoke_out = OUTPUT_ROOT / 'smoke' / SPLIT / 'candidates'
smoke_artifacts = OUTPUT_ROOT / 'smoke' / SPLIT / 'artifacts'
smoke_out.mkdir(parents=True, exist_ok=True); smoke_artifacts.mkdir(parents=True, exist_ok=True)
common_sparse = ['--fields','name','--top-k','50','--chunk-size',str(SPARSE_CHUNK_SIZE),
                 '--max-memory-fraction',str(MAX_MEMORY_FRACTION)]

import polars as pl
def assert_smoke_scope(path):
    returned = set(pl.scan_parquet(path).select(pl.col('query_id').cast(pl.Utf8)).unique().collect()['query_id'].to_list())
    unexpected = returned - set(smoke_ids)
    if unexpected:
        raise RuntimeError(f'Smoke artifact contains rows for unselected queries: {list(unexpected)[:10]}')

smoke_specs = {
 'exact': ('scripts/03_exact_blocking.py', smoke_out/f'{SPLIT}_exact_candidates.parquet',
           [PYTHON, REPO_ROOT/'scripts/03_exact_blocking.py','--data-dir',smoke_data_dir,'--split',SPLIT,'--output-dir',smoke_out,'--artifacts-dir',smoke_artifacts],
           {'semantics':'name_norm/address_norm exact legacy'}),
 'char': ('scripts/04a_char_tfidf_blocking.py', smoke_out/f'{SPLIT}_char_candidates_name_char35_K50.parquet',
          [PYTHON, REPO_ROOT/'scripts/04a_char_tfidf_blocking.py','--data-dir',smoke_data_dir,'--split',SPLIT,'--output-dir',smoke_out,'--artifacts-dir',smoke_artifacts,
           '--ngram-min','3','--ngram-max','5',*common_sparse],
          {'fields':'name','analyzer':'char_wb','ngram_range':[3,5],'max_df':0.01,'min_df':2,'sublinear_tf':True,'k':50}),
 'word': ('scripts/04b_bm25_blocking.py', smoke_out/f'{SPLIT}_bm25_candidates_name_word_K50.parquet',
          [PYTHON, REPO_ROOT/'scripts/04b_bm25_blocking.py','--data-dir',smoke_data_dir,'--split',SPLIT,'--output-dir',smoke_out,'--artifacts-dir',smoke_artifacts,
           '--max-df','0.001','--min-df','2',*common_sparse],
          {'fields':'name','algorithm':'word_tfidf_not_bm25','ngram_range':[1,2],'max_df':0.001,'min_df':2,'sublinear_tf':True,'k':50}),
 'structured': ('scripts/04c_structured_blocking.py', smoke_out/f'{SPLIT}_structured_candidates.parquet',
          [PYTHON, REPO_ROOT/'scripts/04c_structured_blocking.py','--data-dir',smoke_data_dir,'--split',SPLIT,'--output-dir',smoke_out,'--artifacts-dir',smoke_artifacts,
           '--rare-token-max-freq','500','--max-estimated-block-pairs',str(MAX_STRUCTURED_BLOCK_PAIRS)],
          {'legacy_rules':'unchanged','rare_token_max_freq':500,'max_estimated_block_pairs':MAX_STRUCTURED_BLOCK_PAIRS}),
}
toggles = {'exact':RUN_EXACT,'char':RUN_CHAR,'word':RUN_WORD,'structured':RUN_STRUCTURED}
if RUN_SMOKE:
    for retriever, enabled in toggles.items():
        if enabled:
            script, output, command, config = smoke_specs[retriever]
            execute_candidate_stage(label=f'smoke_{SPLIT}_{retriever}', retriever=retriever, script=script,
                command=command, output=output, config=config, input_report=smoke_report)
            assert_smoke_scope(output)

In [ ]:
# Dense corpus embedding/index artifacts are built once from the full S2/S3 corpus; smoke
# then encodes/searches only the selected Source-1 queries. Query caches are kept separate.
dense_dir = OUTPUT_ROOT / 'dense_artifacts' / SPLIT
dense_dir.mkdir(parents=True, exist_ok=True)
dense_force_args = ['--force-rebuild'] if FORCE_REBUILD else []
dense_config = {'model':'all-MiniLM-L6-v2','preprocessing':'name_norm | address_norm | country',
                'index_type':'IVF-SQ8','metric':'inner_product','nlist':16384,'nprobe':32,'k':50,
                'training_sample_method':'first_rows_legacy'}
if RUN_DENSE and (RUN_SMOKE or RUN_FULL):
    run_logged([PYTHON, REPO_ROOT/'scripts/05c_build_dense_index.py','--data-dir',DATA_DIR,'--split',SPLIT,
                '--index-dir',dense_dir,'--model-name','all-MiniLM-L6-v2',*dense_force_args], f'{SPLIT}_dense_embeddings.log')
if RUN_DENSE and RUN_SMOKE:
    dense_smoke_output = smoke_out / f'{SPLIT}_dense_candidates_K50.parquet'
    dense_command = [PYTHON, REPO_ROOT/'scripts/05d_query_dense_index.py','--data-dir',smoke_data_dir,'--split',SPLIT,
        '--index-dir',dense_dir,'--query-cache-dir',dense_dir/'queries_smoke','--output-dir',smoke_out,
        '--model-name','all-MiniLM-L6-v2','--top-k','50','--nlist','16384','--nprobe','32',*dense_force_args]
    execute_candidate_stage(label=f'smoke_{SPLIT}_dense', retriever='dense', script='scripts/05c_build_dense_index.py + scripts/05d_query_dense_index.py',
        command=dense_command, output=dense_smoke_output, config=dense_config, input_report=smoke_report, dense=dense_config)
    assert_smoke_scope(dense_smoke_output)

## 7. Full structured diagnostics safety gate

In [ ]:
if RUN_FULL and RUN_STRUCTURED:
    run_logged([PYTHON, REPO_ROOT/'scripts/04c_structured_blocking.py','--data-dir',DATA_DIR,'--split',SPLIT,
        '--output-dir',OUTPUT_ROOT/SPLIT,'--artifacts-dir',OUTPUT_ROOT/'logs','--rare-token-max-freq','500',
        '--max-estimated-block-pairs',str(MAX_STRUCTURED_BLOCK_PAIRS),'--diagnostics-only'],
        f'{SPLIT}_structured_diagnostics.log')

## 8. Full train/test regeneration — gated on all five smoke manifests

In [ ]:
required_smoke = [OUTPUT_ROOT/'manifests'/f'smoke_{SPLIT}_{name}.json' for name in ('exact','char','word','structured','dense')]
if RUN_FULL:
    missing_smoke = [str(path) for path in required_smoke if not path.is_file()]
    if missing_smoke:
        raise RuntimeError('All five smoke stages must pass before full execution. Missing: '+str(missing_smoke))
    if SPLIT == 'test':
        required_train = [OUTPUT_ROOT/'manifests'/f'train_{name}.json' for name in ('exact','char','word','structured','dense')]
        missing_train = [str(path) for path in required_train if not path.is_file()]
        if missing_train:
            raise RuntimeError('Complete train regeneration before test. Missing: '+str(missing_train))

full_out = OUTPUT_ROOT / SPLIT
full_specs = {name:(script, full_out/output.name,
    [str(DATA_DIR) if str(item)==str(smoke_data_dir) else str(full_out) if str(item)==str(smoke_out)
     else str(OUTPUT_ROOT/'logs') if str(item)==str(smoke_artifacts) else item for item in command], config)
    for name,(script,output,command,config) in smoke_specs.items()}
if RUN_FULL:
    for retriever, enabled in toggles.items():
        if enabled:
            script, output, command, config = full_specs[retriever]
            execute_candidate_stage(label=f'{SPLIT}_{retriever}', retriever=retriever, script=script,
                command=command, output=output, config=config, input_report=processed_report)

In [ ]:
if RUN_FULL and RUN_DENSE:
    dense_output = full_out / f'{SPLIT}_dense_candidates_K50.parquet'
    dense_command = [PYTHON, REPO_ROOT/'scripts/05d_query_dense_index.py','--data-dir',DATA_DIR,'--split',SPLIT,
        '--index-dir',dense_dir,'--query-cache-dir',dense_dir/'queries_full','--output-dir',full_out,
        '--model-name','all-MiniLM-L6-v2','--top-k','50','--nlist','16384','--nprobe','32',*dense_force_args]
    execute_candidate_stage(label=f'{SPLIT}_dense', retriever='dense', script='scripts/05c_build_dense_index.py + scripts/05d_query_dense_index.py',
        command=dense_command, output=dense_output, config=dense_config, input_report=processed_report, dense=dense_config)

## 9. Final validation and Kaggle output inventory

In [ ]:
expected = {
 'exact': full_out/f'{SPLIT}_exact_candidates.parquet',
 'char': full_out/f'{SPLIT}_char_candidates_name_char35_K50.parquet',
 'word': full_out/f'{SPLIT}_bm25_candidates_name_word_K50.parquet',
 'structured': full_out/f'{SPLIT}_structured_candidates.parquet',
 'dense': full_out/f'{SPLIT}_dense_candidates_K50.parquet',
}
if RUN_FULL:
    final_validation = {name:validate_candidate(path, name) for name,path in expected.items()}
    print(json.dumps(final_validation, indent=2))
inventory = []
for path in sorted(OUTPUT_ROOT.rglob('*')):
    if path.is_file():
        inventory.append({'path':str(path.relative_to(OUTPUT_ROOT)), 'bytes':path.stat().st_size})
atomic_json(OUTPUT_ROOT/'artifact_inventory.json', {'preserve_directory':str(OUTPUT_ROOT), 'files':inventory})
print(f'PRESERVE THIS KAGGLE OUTPUT DIRECTORY: {OUTPUT_ROOT}')
print(json.dumps(inventory, indent=2))
print('STOP: canonical A2 union was not run.')